In [ ]:
# !pip install implicit
!pip install lightfm-next

In [ ]:
import json
import pandas as pd
import random
from tqdm import tqdm
from google.colab import drive
import matplotlib.pyplot as plt
from scipy.sparse import coo_matrix

SEED = 22
drive.mount('/content/drive')
import numpy as np
pd.options.display.max_rows = 50
pd.options.display.max_columns = 50

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
csv_file_path = '/content/drive/MyDrive/final_yelp_PA_FL.parquet'
df = pd.read_parquet(csv_file_path)
df

,review_id,user_id,business_id,stars,useful,funny,cool,text,date,name,address,city,state,postal_code,latitude,longitude,stars_business,review_count,is_open,attributes,categories,hours,name_user,review_count_user,yelping_since,useful_user,funny_user,cool_user,elite,friends,fans,average_stars,compliment_hot,compliment_more,compliment_profile,compliment_cute,compliment_list,compliment_note,compliment_plain,compliment_cool,compliment_funny,compliment_writer,compliment_photos,checkin_count,compliment_count
0,KU_O5udG6zpxOg-VcAEodg,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3.0,0,0,0,"If you decide to eat here, just be aware it is...",2018-07-07 22:09:11,Turning Point of North Wales,1460 Bethlehem Pike,North Wales,PA,19454,40.210196,-75.223639,3.0,169,1,"{'AcceptsInsurance': None, 'AgesAllowed': None...","Restaurants, Breakfast & Brunch, Food, Juice B...","{'Friday': '7:30-15:0', 'Monday': '7:30-15:0',...",Melanie,33,2016-01-13 17:20:44,32,3,8,,"DS9QBM_NWJz1E279Zrao-A, XdXgIs4i5JFvtJf0rJlWsA...",0,4.06,0,0,0,0,0,0,0,1,1,0,0,177.0,0.0
1,BiTunyQ73aT9WBnpR9DZGw,OyoGAe7OKpv6SyGZT5g77Q,7ATYjTIgM3jUlt4UM3IypQ,5.0,1,0,1,I've taken a lot of spin classes over the year...,2012-01-03 15:28:18,Body Cycle Spinning Studio,"1923 Chestnut St, 2nd Fl",Philadelphia,PA,19119,39.952103,-75.172753,5.0,144,0,"{'AcceptsInsurance': None, 'AgesAllowed': None...","Active Life, Cycling Classes, Trainers, Gyms, ...","{'Friday': '6:0-19:30', 'Monday': '6:30-20:30'...",Erin,10,2011-03-07 19:45:15,6,1,2,,"7uYgWwryg8KH33i1SLJUTQ, 0mGJMNL8o2AY4BT1d4TTDQ...",0,4.30,0,0,0,0,0,0,0,0,0,0,0,497.0,0.0
2,J-4NdnDZ0pUQaUEEwDI9KQ,vrKkXsozqqecF3CW4cGaVQ,rjuWz_AD3WfXJc03AhIO_w,5.0,2,2,2,I thoroughly enjoyed the show. Chill way to s...,2012-12-04 16:46:20,The N Crowd,111 S Independence Mall E,Philadelphia,PA,19106,39.949756,-75.148062,4.5,90,1,"{'AcceptsInsurance': None, 'AgesAllowed': None...","Performing Arts, Arts & Entertainment, Nightli...","{'Friday': '19:15-21:15', 'Monday': '0:0-0:0',...",Mike,120,2010-08-16 19:59:54,48,17,31,,"ZaUT63HFjheiub1y7019Yg, Obkepp5aBTYVX-AL4zwBoQ...",5,4.44,4,3,0,0,0,10,15,4,4,4,0,148.0,0.0
3,DyrAIuKl60j_X8Yrrv-kpg,mNsVyC9tQVYtzLOCbh2Piw,MWmXGQ98KbRo3vsS5nZhMA,5.0,1,0,0,I recently had dinner here with my wife over t...,2014-10-27 02:47:28,Anthony's at Paxon Hollow,850 Paxon Hollow Rd,Broomall,PA,19008,39.958108,-75.371118,3.5,32,1,"{'AcceptsInsurance': None, 'AgesAllowed': None...","Event Planning & Services, Italian, Venues & E...","{'Friday': '12:0-21:0', 'Monday': None, 'Satur...",Michael,12,2014-07-20 22:56:07,43,6,2,,RKULSOrIvvYpDmtuYXEXzA,0,2.53,0,0,0,0,0,0,0,0,0,0,0,15.0,0.0
4,40thYphUgIfvJq17QCfTwA,QzCEzH3R7Z6erOGLr3t55Q,0pMj5xUAecW9o1P35B0AMw,5.0,1,0,1,Great staff always helps and always nice. Alwa...,2017-05-26 13:10:24,Wawa,2544 W Main Street,Norristown,PA,19403,40.141292,-75.389286,3.5,8,1,"{'AcceptsInsurance': None, 'AgesAllowed': None...","Food, Coffee & Tea, Gas Stations, Restaurants,...","{'Friday': '0:0-0:0', 'Monday': '0:0-0:0', 'Sa...",Kylhalil,14,2012-03-05 19:12:11,5,2,2,,"POmz_WtE-nowjX5H7s9NSA, PEYAcshPlYwY-YP-55Re4Q...",0,4.36,0,0,0,0,0,0,0,0,0,0,0,79.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
673685,rtt1Ymczj-1Lb26JMsY2lA,M1cMsRL4L7IUr9RILDywEQ,vt_esoDw6HG5ClM12OPkMg,4.0,4,3,4,"5 stars for the Bonte waffle, 3 stars for this...",2009-03-03 20:59:10,Bonté Wafflerie & Café,130 S 17th St,Philadelphia,PA,19102,39.950510,-75.169205,3.5,115,0,"{'AcceptsInsurance': None, 'AgesAllowed': None...","Coffee & Tea, Restaurants, Breakfast & Brunch,...","{'Friday': '6:30-18:0', 'Monday': '6:30-18:0',...",Matt,2227,2006-10-24 22:40:11,5681,1342,2430,"2008,2009,2010,2011,2012,2013,2014,2015,2016,2...","TLElemg3o3mlA4GGpupNiQ, REGjppkq2IkDLRQVmRj_Bg...",277,3.78,59,30,10,13,36,79,116,150,150,128,41,193.0,0.0
673686,5n_oSwXspiiSsZgNwjp48g,bJ5FtCtZX3ZZacz2_2PJjA,SOsjW1JARmtHUFtpFlp8rw,4.0,5,

In [ ]:
import numpy as np
import pandas as pd
from scipy.sparse import coo_matrix

SEED = 42
np.random.seed(SEED)

MIN_USER_INTERACTIONS = 5
MIN_ITEM_INTERACTIONS = 5
K_TEST = 1
K_VAL  = 1

assert MIN_USER_INTERACTIONS >= (K_TEST + K_VAL + 1), "иначе train_inner может стать пустым"

df0 = df.copy()
df0["date"] = pd.to_datetime(df0["date"], errors="coerce")
df0 = (df0
       .dropna(subset=["user_id", "business_id", "date"])
       .sort_values(["user_id", "date"])
)

user_cnt = df0.groupby("user_id")["business_id"].size()
eligible_users = user_cnt[user_cnt >= MIN_USER_INTERACTIONS].index
df_cf = df0[df0["user_id"].isin(eligible_users)].copy()

print("Users after filter:", df_cf["user_id"].nunique())
print("Total interactions:", len(df_cf))


Users after filter: 40711
Total interactions: 519891


## Data split

In [ ]:
# test = last K_TEST per user
test_idx = df_cf.groupby("user_id").tail(K_TEST).index
test0_full  = df_cf.loc[test_idx].copy()
train0_full = df_cf.drop(test_idx).copy()

# val = last K_VAL per user inside train0
val_idx = train0_full.groupby("user_id").tail(K_VAL).index
val0_full        = train0_full.loc[val_idx].copy()
train_inner0_full = train0_full.drop(val_idx).copy()

print("train_inner0:", len(train_inner0_full), "val0:", len(val0_full), "test0:", len(test0_full))
print("min interactions per user in train_inner0:",
      int(train_inner0_full.groupby("user_id")["business_id"].size().min()))

# минимальные версии (как в твоём CF-паке)
train0       = train0_full[["user_id","business_id","date"]].copy()
test0        = test0_full[["user_id","business_id","date"]].copy()
val0         = val0_full[["user_id","business_id","date"]].copy()
train_inner0 = train_inner0_full[["user_id","business_id","date"]].copy()


train_inner0: 438469 val0: 40711 test0: 40711
min interactions per user in train_inner0: 3


In [ ]:
def build_split_objects(train_df, eval_df, min_item_interactions=5):
    item_cnt = train_df.groupby("business_id")["user_id"].size()
    eligible_items = item_cnt[item_cnt >= min_item_interactions].index

    train_f = train_df[train_df["business_id"].isin(eligible_items)].copy()
    eval_f  = eval_df[eval_df["business_id"].isin(eligible_items)].copy()

    users = train_f["user_id"].unique()
    items = train_f["business_id"].unique()
    user2idx = {u:i for i,u in enumerate(users)}
    item2idx = {b:i for i,b in enumerate(items)}

    train_f["u"] = train_f["user_id"].map(user2idx).astype("int32")
    train_f["i"] = train_f["business_id"].map(item2idx).astype("int32")

    eval_f["u"] = eval_f["user_id"].map(user2idx)
    eval_f["i"] = eval_f["business_id"].map(item2idx)

    before = len(eval_f)
    eval_f = eval_f.dropna(subset=["u","i"]).copy()
    eval_f["u"] = eval_f["u"].astype("int32")
    eval_f["i"] = eval_f["i"].astype("int32")
    dropped = before - len(eval_f)

    R = coo_matrix(
        (np.ones(len(train_f), dtype=np.float32),
         (train_f["u"].values, train_f["i"].values)),
        shape=(len(users), len(items))
    ).tocsr()

    user_train_items = train_f.groupby("u")["i"].apply(lambda x: set(x.tolist())).to_dict()
    all_items = np.arange(len(items), dtype=np.int32)

    return {
        "n_users": len(users),
        "n_items": len(items),
        "R": R,
        "train_f": train_f,
        "eval_f": eval_f,
        "eval_u": eval_f["u"].values,
        "eval_i": eval_f["i"].values,
        "user_train_items": user_train_items,
        "all_items": all_items,
        "dropped_cold_start": dropped,
        "eligible_items": eligible_items  # полезно для CB
    }

val_pack  = build_split_objects(train_inner0, val0,  min_item_interactions=MIN_ITEM_INTERACTIONS)
test_pack = build_split_objects(train0,       test0, min_item_interactions=MIN_ITEM_INTERACTIONS)

print("VAL: n_users/n_items:", val_pack["n_users"], val_pack["n_items"],
      "| eval:", len(val_pack["eval_u"]), "| dropped cold:", val_pack["dropped_cold_start"])
print("TEST: n_users/n_items:", test_pack["n_users"], test_pack["n_items"],
      "| eval:", len(test_pack["eval_u"]), "| dropped cold:", test_pack["dropped_cold_start"])


VAL: n_users/n_items: 40593 18699 | eval: 32656 | dropped cold: 55
TEST: n_users/n_items: 40666 19983 | eval: 32032 | dropped cold: 25


In [ ]:
def agg_item_text(train_full: pd.DataFrame, eligible_items: pd.Index) -> pd.Series:
    x = train_full[train_full["business_id"].isin(eligible_items)][["business_id","text"]].copy()
    x["text"] = x["text"].fillna("").astype(str)
    return x.groupby("business_id")["text"].apply(lambda s: " ".join(t for t in s.values if t))

# ВАЖНО: без утечки
# - для VAL: только train_inner0_full
# - для TEST: train0_full (train_inner + val)
item_text_val  = agg_item_text(train_inner0_full, val_pack["eligible_items"])
item_text_test = agg_item_text(train0_full,       test_pack["eligible_items"])

print("item_text_val items:",  item_text_val.shape[0])
print("item_text_test items:", item_text_test.shape[0])


item_text_val items: 18699
item_text_test items: 19983


## Metrics

In [ ]:
import numpy as np

def _ndcg_from_rank(rank: int) -> float:
    # rank: 0-based
    return 1.0 / np.log2(rank + 2)

def _ap_from_rank(rank: int) -> float:
    # AP для single-positive случая (LMROO-1) = 1/(rank+1)
    return 1.0 / (rank + 1)

def _extract_items(recs):
    """
    Нормализуем вывод recommend_fn:
    - если вернул np.array/list -> это items
    - если вернул (items, scores) -> берём items
    """
    if recs is None:
        return np.array([], dtype=np.int32)
    if isinstance(recs, tuple) and len(recs) >= 1:
        recs = recs[0]
    return np.asarray(recs, dtype=np.int32)

def evaluate_next_item(eval_u, eval_i, recommend_fn, ks=(5,10,20,50), name="model"):
    """
    Оценка next-item (по одному target на пользователя).
    ВАЖНО: ничего дополнительно из eval не выкидываем (в т.ч. если target item был seen ранее).
    """
    ks = tuple(sorted(ks))
    maxK = ks[-1]
    n = len(eval_u)
    if n == 0:
        print(f"[{name}] empty eval set")
        return {k: (0.0, 0.0, 0.0) for k in ks}

    rec = {k: 0.0 for k in ks}
    ndcg = {k: 0.0 for k in ks}
    mp = {k: 0.0 for k in ks}
    short_lists = 0

    for u, pos in zip(eval_u, eval_i):
        items = _extract_items(recommend_fn(int(u), int(maxK)))

        if len(items) < maxK:
            short_lists += 1

        # позиция target в рекомендованном списке (0-based)
        rpos = None
        for r, it in enumerate(items):
            if int(it) == int(pos):
                rpos = r
                break

        for k in ks:
            if rpos is None or rpos >= k:
                continue
            rec[k] += 1.0
            ndcg[k] += _ndcg_from_rank(rpos)
            mp[k] += _ap_from_rank(rpos)

    out = {k: (rec[k]/n, ndcg[k]/n, mp[k]/n) for k in ks}
    print(f"[{name}] short rec lists: {short_lists}/{n} ({short_lists/n:.3f})")
    return out

def print_metrics(name, out, ks=(5,10,20,50)):
    print(f"\n=== {name} ===")
    for k in ks:
        r, n, m = out[k]
        print(f"K={k:2d} | Recall@{k}: {r:.4f} | NDCG@{k}: {n:.4f} | MAP@{k}: {m:.4f}")


## Feature engeneering

In [ ]:
# Берём статические поля бизнеса (это не даёт утечки)
need_item_cols = [c for c in ["business_id","categories","attributes","stars_business","review_count","is_open"]
                  if c in df_cf.columns]

items_meta = (df_cf[need_item_cols]
              .sort_values(["business_id"])
              .groupby("business_id", as_index=False)
              .first())


In [ ]:
import re, ast

_re_tok = re.compile(r"[^a-z0-9]+", re.IGNORECASE)

def _clean_token(s: str) -> str:
    s = str(s).strip().lower()
    s = s.replace("&", " and ")
    s = _re_tok.sub("_", s).strip("_")
    return s

def _cat_tokens(cat):
    if pd.isna(cat) or cat is None:
        return []
    parts = [p.strip() for p in str(cat).split(",") if p.strip()]
    return [f"cat_{_clean_token(p)}" for p in parts]

def _safe_literal_eval(x):
    if isinstance(x, dict):
        return x
    if pd.isna(x) or x is None:
        return {}
    s = str(x)
    # убираем префиксы u'...' / u"..."
    s = s.replace("u'", "'").replace('u"', '"')
    try:
        return ast.literal_eval(s)
    except Exception:
        return {}

def _norm_scalar(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return None
    if isinstance(v, bool):
        return "true" if v else "false"
    s = str(v).strip()
    s = s.replace("u'", "'").replace('u"', '"')
    # снимем внешние кавычки типа "'loud'"
    if len(s) >= 2 and ((s[0] == s[-1] == "'") or (s[0] == s[-1] == '"')):
        s = s[1:-1].strip()
    return s if s else None

def _attr_tokens(attr):
    d = _safe_literal_eval(attr)
    out = []
    for k, v in (d or {}).items():
        if v is None:
            continue
        key = _clean_token(k)

        # вложенные dict часто лежат строкой -> распарсим ещё раз
        if isinstance(v, str) and v.strip().startswith("{") and v.strip().endswith("}"):
            v2 = _safe_literal_eval(v)
            if isinstance(v2, dict):
                v = v2

        if isinstance(v, dict):
            for kk, vv in v.items():
                vv = _norm_scalar(vv)
                if vv is None:
                    continue
                if str(vv).lower() in ("true", "false"):
                    if str(vv).lower() == "true":
                        out.append(f"attr_{key}_{_clean_token(kk)}")
                else:
                    out.append(f"attr_{key}_{_clean_token(kk)}_{_clean_token(vv)}")
        else:
            vv = _norm_scalar(v)
            if vv is None:
                continue
            if str(vv).lower() in ("true", "false"):
                if str(vv).lower() == "true":
                    out.append(f"attr_{key}")
            else:
                out.append(f"attr_{key}_{_clean_token(vv)}")
    return out

def build_item_docs_for_pack(pack, items_meta, item_text_series):
    """
    Возвращает:
      idx2biz: np.array business_id в порядке i=0..n_items-1 (строго как в pack)
      docs:    list[str] документов (categories+attributes+review_text_agg + числовые токены)
    """
    idx2biz = (pack["train_f"][["i","business_id"]]
               .drop_duplicates("i")
               .sort_values("i")["business_id"].to_numpy())

    meta = items_meta.set_index("business_id")
    docs = []

    for b in idx2biz:
        row = meta.loc[b] if b in meta.index else None

        cats = _cat_tokens(row["categories"]) if row is not None and "categories" in meta.columns else []
        attrs = _attr_tokens(row["attributes"]) if row is not None and "attributes" in meta.columns else []

        # ДОБАВЛЕНО: числовые/булевы статические поля бизнеса -> токены
        extra = []
        if row is not None:
            if "stars_business" in meta.columns and pd.notna(row["stars_business"]):
                # дискретизируем по 0.5: 4.5 -> 45
                sb = int(round(float(row["stars_business"]) * 10))
                extra.append(f"num_starsbiz_{sb}")
            if "review_count" in meta.columns and pd.notna(row["review_count"]):
                # бинирование по лог-шкале (грубо, но компактно)
                rc = int(row["review_count"])
                bin_rc = int(np.clip(np.floor(np.log10(rc + 1) * 10), 0, 50))
                extra.append(f"num_reviewcountbin_{bin_rc}")
            if "is_open" in meta.columns and pd.notna(row["is_open"]):
                extra.append("flag_open" if int(row["is_open"]) == 1 else "flag_closed")

        # rev = item_text_series.get(b, "")  # item_text_* уже без утечки

        docs.append(" ".join(cats + attrs + extra))

    return idx2biz, docs


idx2biz_val,  item_docs_val  = build_item_docs_for_pack(val_pack,  items_meta, item_text_val)
idx2biz_test, item_docs_test = build_item_docs_for_pack(test_pack, items_meta, item_text_test)

print("VAL docs:", len(item_docs_val), "TEST docs:", len(item_docs_test))


VAL docs: 18699 TEST docs: 19983


In [ ]:
item_docs_val[2]

'cat_breweries cat_food cat_american_new cat_beer cat_wine_and_spirits cat_restaurants cat_seafood attr_alcohol_full_bar attr_ambience_trendy attr_ambience_classy attr_bikeparking attr_businessacceptscreditcards attr_businessparking_street attr_businessparking_lot attr_businessparking_valet attr_dogsallowed attr_goodforkids attr_goodformeal_lunch attr_goodformeal_dinner attr_noiselevel_average attr_outdoorseating attr_restaurantsattire_casual attr_restaurantsdelivery_none attr_restaurantsgoodforgroups attr_restaurantspricerange2_2 attr_restaurantsreservations attr_restaurantstableservice attr_restaurantstakeout attr_wheelchairaccessible attr_wifi_free num_starsbiz_40 num_reviewcountbin_34 flag_open'

## LightFM

In [ ]:
import numpy as np
from scipy.sparse import identity, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sklearn.model_selection import ParameterGrid

from lightfm import LightFM
NUM_THREADS = 8          # поставь по CPU
VAL_SAMPLE = 300000        # 1000-5000 для быстрого тюнинга
TARGET_K = 20
KS = (5, 10, 20, 50)

rng = np.random.default_rng(SEED)
idx = np.arange(len(val_pack["eval_u"]))
if len(idx) > VAL_SAMPLE:
    idx = rng.choice(idx, VAL_SAMPLE, replace=False)
val_u_small = val_pack["eval_u"][idx]
val_i_small = val_pack["eval_i"][idx]



def build_item_features(pack, item_docs, tfidf_params):
    vec = TfidfVectorizer(**tfidf_params)
    X = vec.fit_transform(item_docs).astype(np.float32)
    X = normalize(X, norm="l2", axis=1)

    I = identity(pack["n_items"], format="csr", dtype=np.float32)
    item_features = hstack([I, X], format="csr")  # (n_items, n_items + n_terms)
    return item_features

def train_lfm(pack, item_docs, tfidf_params, lfm_params, epochs):
    interactions = pack["R"].tocsr().astype(np.float32)

    item_features = build_item_features(pack, item_docs, tfidf_params)
    user_features = identity(pack["n_users"], format="csr", dtype=np.float32)

    model = LightFM(**lfm_params)
    model.fit(
        interactions,
        user_features=user_features,
        item_features=item_features,
        epochs=int(epochs),
        num_threads=NUM_THREADS
    )
    return model, user_features, item_features

TFIDF_PARAMS = dict(
    lowercase=True,
    stop_words="english",
    norm="l2",
    sublinear_tf=True,
    ngram_range=(1, 1),     # быстро; биграммы потом, если надо
    min_df=2,
    max_df=0.9,
    max_features=300_000
)
candidates = [
    {
        "name": "TOP-1 (№28)",
        "tfidf": {**TFIDF_PARAMS, "min_df": 5, "ngram_range": (1, 2), "max_features": 200_000},
        "lfm": {"no_components": 64, "item_alpha": 1e-06, "loss": "warp", "random_state": SEED}
    },
    {
        "name": "TOP-2 (№12)",
        "tfidf": {**TFIDF_PARAMS, "min_df": 2, "ngram_range": (1, 2), "max_features": 200_000},
        "lfm": {"no_components": 64, "item_alpha": 1e-06, "loss": "warp", "random_state": SEED}
    },
    {
        "name": "TOP-3 (№16)",
        "tfidf": {**TFIDF_PARAMS, "min_df": 2, "ngram_range": (1, 2), "max_features": 200_000},
        "lfm": {"no_components": 64, "item_alpha": 1e-05, "loss": "warp", "random_state": SEED}
    }
]
def make_lfm_recommender(pack, model, user_features, item_features, filter_seen=True):
    all_items = pack["all_items"]
    user_train_items = pack["user_train_items"]
    cache = {}

    def recommend_fn(u: int, K: int):
        if u not in cache:
            scores = model.predict(
                user_ids=u,
                item_ids=all_items,
                user_features=user_features,
                item_features=item_features,
                num_threads=NUM_THREADS
            ).astype(np.float32)

            if filter_seen:
                seen = user_train_items.get(u, None)
                if seen:
                    scores[list(seen)] = -np.inf
            cache[u] = scores

        scores = cache[u]
        K = min(int(K), scores.shape[0])
        if K <= 0:
            return np.array([], dtype=np.int32)

        idx = np.argpartition(-scores, K-1)[:K]
        idx = idx[np.argsort(-scores[idx])]
        return idx.astype(np.int32)

    return recommend_fn


EPOCHS = 25
results = {}

for cand in candidates:
    print(f"\n>>> Evaluating {cand['name']}...")

    # 1. Обучаем на VAL для честной проверки метрик
    model, uf, itf = train_lfm(
        val_pack,
        item_docs_val,
        cand["tfidf"],
        cand["lfm"],
        epochs=EPOCHS
    )

    # 2. Считаем метрики на полном валидационном сете
    rec_fn = make_lfm_recommender(val_pack, model, uf, itf, filter_seen=True)
    val_out = evaluate_next_item(
        val_pack["eval_u"],
        val_pack["eval_i"],
        rec_fn,
        ks=KS,
        name=cand["name"]
    )

    results[cand["name"]] = val_out
    print_metrics(f"FINAL VAL: {cand['name']}", val_out, ks=KS)

# --- Итоговое сравнение NDCG@20 ---
print("\n" + "="*30)
print("FINAL COMPARISON (NDCG@20)")
print("="*30)
for name, res in results.items():
    print(f"{name}: {res[20][1]:.6f}")


>>> Evaluating TOP-1 (№28)...
[TOP-1 (№28)] short rec lists: 0/32656 (0.000)

=== FINAL VAL: TOP-1 (№28) ===
K= 5 | Recall@5: 0.0123 | NDCG@5: 0.0075 | MAP@5: 0.0060
K=10 | Recall@10: 0.0216 | NDCG@10: 0.0105 | MAP@10: 0.0072
K=20 | Recall@20: 0.0376 | NDCG@20: 0.0145 | MAP@20: 0.0083
K=50 | Recall@50: 0.0716 | NDCG@50: 0.0212 | MAP@50: 0.0093

>>> Evaluating TOP-2 (№12)...
[TOP-2 (№12)] short rec lists: 0/32656 (0.000)

=== FINAL VAL: TOP-2 (№12) ===
K= 5 | Recall@5: 0.0117 | NDCG@5: 0.0073 | MAP@5: 0.0059
K=10 | Recall@10: 0.0213 | NDCG@10: 0.0104 | MAP@10: 0.0072
K=20 | Recall@20: 0.0364 | NDCG@20: 0.0142 | MAP@20: 0.0082
K=50 | Recall@50: 0.0732 | NDCG@50: 0.0214 | MAP@50: 0.0093

>>> Evaluating TOP-3 (№16)...
[TOP-3 (№16)] short rec lists: 0/32656 (0.000)

=== FINAL VAL: TOP-3 (№16) ===
K= 5 | Recall@5: 0.0130 | NDCG@5: 0.0079 | MAP@5: 0.0063
K=10 | Recall@10: 0.0216 | NDCG@10: 0.0106 | MAP@10: 0.0074
K=20 | Recall@20: 0.0368 | NDCG@20: 0.0145 | MAP@20: 0.0084
K=50 | Recall@50: 0

In [ ]:
import numpy as np
from scipy.sparse import identity, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sklearn.model_selection import ParameterGrid

from lightfm import LightFM

# --- настройки скорости ---
NUM_THREADS = 8          # поставь по CPU
VAL_SAMPLE = 300000        # 1000-5000 для быстрого тюнинга
TARGET_K = 20
KS = (5, 10, 20, 50)

rng = np.random.default_rng(SEED)
idx = np.arange(len(val_pack["eval_u"]))
if len(idx) > VAL_SAMPLE:
    idx = rng.choice(idx, VAL_SAMPLE, replace=False)
val_u_small = val_pack["eval_u"][idx]
val_i_small = val_pack["eval_i"][idx]

# --- TF-IDF параметры (фиксируем для тюнинга LightFM; можно потом расширить) ---
TFIDF_PARAMS = dict(
    lowercase=True,
    stop_words="english",
    norm="l2",
    sublinear_tf=True,
    ngram_range=(1, 1),     # быстро; биграммы потом, если надо
    min_df=2,
    max_df=0.9,
    max_features=300_000
)

def build_item_features(pack, item_docs, tfidf_params):
    vec = TfidfVectorizer(**tfidf_params)
    X = vec.fit_transform(item_docs).astype(np.float32)
    X = normalize(X, norm="l2", axis=1)

    I = identity(pack["n_items"], format="csr", dtype=np.float32)
    item_features = hstack([I, X], format="csr")  # (n_items, n_items + n_terms)
    return item_features

def train_lfm(pack, item_docs, tfidf_params, lfm_params, epochs):
    interactions = pack["R"].tocsr().astype(np.float32)

    item_features = build_item_features(pack, item_docs, tfidf_params)
    user_features = identity(pack["n_users"], format="csr", dtype=np.float32)

    model = LightFM(**lfm_params)
    model.fit(
        interactions,
        user_features=user_features,
        item_features=item_features,
        epochs=int(epochs),
        num_threads=NUM_THREADS
    )
    return model, user_features, item_features

def make_lfm_recommender(pack, model, user_features, item_features, filter_seen=True):
    all_items = pack["all_items"]
    user_train_items = pack["user_train_items"]
    cache = {}

    def recommend_fn(u: int, K: int):
        if u not in cache:
            scores = model.predict(
                user_ids=u,
                item_ids=all_items,
                user_features=user_features,
                item_features=item_features,
                num_threads=NUM_THREADS
            ).astype(np.float32)

            if filter_seen:
                seen = user_train_items.get(u, None)
                if seen:
                    scores[list(seen)] = -np.inf
            cache[u] = scores

        scores = cache[u]
        K = min(int(K), scores.shape[0])
        if K <= 0:
            return np.array([], dtype=np.int32)

        idx = np.argpartition(-scores, K-1)[:K]
        idx = idx[np.argsort(-scores[idx])]
        return idx.astype(np.int32)

    return recommend_fn

def ndcg20_on_val_fast(lfm_params, epochs):
    model, uf, itf = train_lfm(val_pack, item_docs_val, TFIDF_PARAMS, lfm_params, epochs)
    rec_fn = make_lfm_recommender(val_pack, model, uf, itf, filter_seen=True)
    out = evaluate_next_item(val_u_small, val_i_small, rec_fn, ks=(TARGET_K,), name="LFM@VAL_tune")
    return out[TARGET_K][1]  # ndcg@20

# ---- Grid Search по LightFM (маленькая сетка = быстро) ----
grid = ParameterGrid({
    "loss": ["warp"],                 # самый популярный для ранжирования
    "no_components": [32, 64, 128],        # факторы
    "learning_rate": [0.05, 0.02],          # можно добавить 0.02 если хочешь
    "item_alpha": [1e-6, 1e-5],
    # "user_alpha": [1e-6, 1e-5],
})
EPOCHS = 25

best_lfm_params, best_ndcg = None, -1.0
i = 0

for p in grid:
    i += 1
    lfm_params = dict(
        random_state=SEED,
        **p
    )
    print(f"[{i}/{len(grid)}] testing:", lfm_params)
    ndcg = ndcg20_on_val_fast(lfm_params, epochs=EPOCHS)
    print("   ndcg@20:", ndcg)

    if ndcg > best_ndcg:
        best_ndcg, best_lfm_params = ndcg, lfm_params

print("\nBest LightFM params:", best_lfm_params, "| tune NDCG@20:", best_ndcg)

# ---- Полный VAL отчёт (один раз) ----
model_val, uf_val, itf_val = train_lfm(val_pack, item_docs_val, TFIDF_PARAMS, best_lfm_params, epochs=EPOCHS)
rec_val = make_lfm_recommender(val_pack, model_val, uf_val, itf_val, filter_seen=True)
val_out = evaluate_next_item(val_pack["eval_u"], val_pack["eval_i"], rec_val, ks=KS, name="LFM@VAL_full")
print_metrics("LFM@VAL_full", val_out, ks=KS)

# ---- Финальная модель на TEST (train0 -> test0) ----
model_test, uf_test, itf_test = train_lfm(test_pack, item_docs_test, TFIDF_PARAMS, best_lfm_params, epochs=EPOCHS)
rec_test = make_lfm_recommender(test_pack, model_test, uf_test, itf_test, filter_seen=True)
test_out = evaluate_next_item(test_pack["eval_u"], test_pack["eval_i"], rec_test, ks=KS, name="LFM@TEST")
print_metrics("LFM@TEST", test_out, ks=KS)


In [ ]:
# (min_df: 2, k: 64, alpha: 1e-06)

In [ ]:
# import numpy as np
# from scipy.sparse import identity, hstack
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.preprocessing import normalize
# from sklearn.model_selection import ParameterGrid
# from lightfm import LightFM

# NUM_THREADS = 8
# VAL_SAMPLE = 50000000
# TARGET_K = 20
# KS = (5, 10, 20, 50)
# EPOCHS_TUNE = 15
# EPOCHS_FINAL = 20

# # --- подвыборка для быстрого тюнинга ---
# rng = np.random.default_rng(SEED)
# idx = np.arange(len(val_pack["eval_u"]))
# if len(idx) > VAL_SAMPLE:
#     idx = rng.choice(idx, VAL_SAMPLE, replace=False)
# val_u_small = val_pack["eval_u"][idx]
# val_i_small = val_pack["eval_i"][idx]

# def build_item_features(pack, item_docs, tfidf_params):
#     vec = TfidfVectorizer(**tfidf_params)
#     X = vec.fit_transform(item_docs).astype(np.float32)
#     X = normalize(X, norm="l2", axis=1)
#     I = identity(pack["n_items"], format="csr", dtype=np.float32)
#     return hstack([I, X], format="csr")

# def make_recommender(pack, model, user_features, item_features, filter_seen=True):
#     all_items = pack["all_items"]
#     user_train_items = pack["user_train_items"]
#     cache = {}

#     def recommend_fn(u: int, K: int):
#         if u not in cache:
#             scores = model.predict(
#                 user_ids=u, item_ids=all_items,
#                 user_features=user_features, item_features=item_features,
#                 num_threads=NUM_THREADS
#             ).astype(np.float32)
#             if filter_seen:
#                 seen = user_train_items.get(u, None)
#                 if seen:
#                     scores[list(seen)] = -np.inf
#             cache[u] = scores

#         scores = cache[u]
#         K = min(int(K), scores.shape[0])
#         if K <= 0:
#             return np.array([], dtype=np.int32)
#         top = np.argpartition(-scores, K-1)[:K]
#         top = top[np.argsort(-scores[top])]
#         return top.astype(np.int32)

#     return recommend_fn

# TFIDF_GRID = ParameterGrid({
#     "ngram_range": [(1, 1), (1,2)],
#     "min_df": [2, 5],
#     "max_df": [0.9],
#     "max_features": [200_000, 300_000],
# })

# LFM_GRID = ParameterGrid({
#     "loss": ["warp"],
#     "no_components": [64],
#     "learning_rate": [0.05, 0.02],
#     "item_alpha": [1e-6, 1e-5],
#     # user_alpha фиксируем равным item_alpha, чтобы не раздувать перебор
# })

# best = {"ndcg": -1.0, "tfidf": None, "lfm": None}
# trial = 0
# total = len(list(TFIDF_GRID)) * len(list(LFM_GRID))

# for tf in TFIDF_GRID:
#     tfidf_params = dict(
#         lowercase=True, stop_words="english", norm="l2",
#         sublinear_tf=True,
#         **tf
#     )

#     # TF-IDF фичи строим один раз на эту конфигурацию
#     item_features_val = build_item_features(val_pack, item_docs_val, tfidf_params)
#     user_features_val = identity(val_pack["n_users"], format="csr", dtype=np.float32)
#     interactions_val = val_pack["R"].tocsr().astype(np.float32)

#     for lf in LFM_GRID:
#         trial += 1
#         lfm_params = dict(
#             random_state=SEED,
#             loss=lf["loss"],
#             no_components=lf["no_components"],
#             learning_rate=lf["learning_rate"],
#             item_alpha=lf["item_alpha"],
#             user_alpha=lf["item_alpha"],   # фиксируем
#         )

#         print(f"[{trial}/{total}] TFIDF={tf} | LFM={{k={lfm_params['no_components']}, alpha={lfm_params['item_alpha']}}}")

#         model = LightFM(**lfm_params)
#         model.fit(
#             interactions_val,
#             user_features=user_features_val,
#             item_features=item_features_val,
#             epochs=EPOCHS_TUNE,
#             num_threads=NUM_THREADS
#         )

#         rec_fn = make_recommender(val_pack, model, user_features_val, item_features_val, filter_seen=True)
#         out = evaluate_next_item(val_u_small, val_i_small, rec_fn, ks=(TARGET_K,), name="LFM@VAL_tune")
#         ndcg = out[TARGET_K][1]
#         print("   ndcg@20:", ndcg)

#         if ndcg > best["ndcg"]:
#             best.update({"ndcg": ndcg, "tfidf": tfidf_params, "lfm": lfm_params})

# print("\nBEST (tune):")
# print("  tfidf:", {k: best["tfidf"][k] for k in ["min_df","max_df","max_features","ngram_range"]})
# print("  lfm  :", {k: best["lfm"][k] for k in ["no_components","item_alpha","user_alpha","learning_rate","loss"]})
# print("  ndcg@20:", best["ndcg"])

# # ---- полный VAL (один раз) ----
# item_features_val = build_item_features(val_pack, item_docs_val, best["tfidf"])
# user_features_val = identity(val_pack["n_users"], format="csr", dtype=np.float32)
# interactions_val = val_pack["R"].tocsr().astype(np.float32)

# model_val = LightFM(**best["lfm"])
# model_val.fit(interactions_val, user_features=user_features_val, item_features=item_features_val,
#               epochs=EPOCHS_FINAL, num_threads=NUM_THREADS)
# rec_val = make_recommender(val_pack, model_val, user_features_val, item_features_val, filter_seen=True)
# val_out = evaluate_next_item(val_pack["eval_u"], val_pack["eval_i"], rec_val, ks=KS, name="LFM@VAL_full")
# print_metrics("LFM@VAL_full", val_out, ks=KS)

# # ---- TEST (один раз, с теми же гиперами) ----
# item_features_test = build_item_features(test_pack, item_docs_test, best["tfidf"])
# user_features_test = identity(test_pack["n_users"], format="csr", dtype=np.float32)
# interactions_test = test_pack["R"].tocsr().astype(np.float32)

# model_test = LightFM(**best["lfm"])
# model_test.fit(interactions_test, user_features=user_features_test, item_features=item_features_test,
#                epochs=EPOCHS_FINAL, num_threads=NUM_THREADS)
# rec_test = make_recommender(test_pack, model_test, user_features_test, item_features_test, filter_seen=True)
# test_out = evaluate_next_item(test_pack["eval_u"], test_pack["eval_i"], rec_test, ks=KS, name="LFM@TEST")
# print_metrics("LFM@TEST", test_out, ks=KS)


[1/32] TFIDF={'max_df': 0.9, 'max_features': 200000, 'min_df': 2, 'ngram_range': (1, 1)} | LFM={k=64, alpha=1e-06}


In [ ]:
import gc
import numpy as np
from scipy.sparse import identity, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sklearn.model_selection import ParameterGrid
from lightfm import LightFM

NUM_THREADS = 8
VAL_SAMPLE = 10_000   # если это опечатка — уменьши, но код теперь не делает np.arange на весь размер
TARGET_K = 20
KS = (5, 10, 20, 50)
EPOCHS_TUNE = 15
EPOCHS_FINAL = 20

def sample_eval_pairs(pack, sample_n, seed):
    """Семплинг без np.arange(len(...)) — меньше риск OOM."""
    n = len(pack["eval_u"])
    if (sample_n is None) or (sample_n >= n):
        return pack["eval_u"], pack["eval_i"]
    rng = np.random.default_rng(seed)
    idx = rng.choice(n, size=int(sample_n), replace=False)
    return pack["eval_u"][idx], pack["eval_i"][idx]

val_u_small, val_i_small = sample_eval_pairs(val_pack, VAL_SAMPLE, SEED)

def build_item_features(pack, item_docs, tfidf_params):
    if len(item_docs) != pack["n_items"]:
        raise ValueError(
            f"len(item_docs)={len(item_docs)} != n_items={pack['n_items']} "
            f"(проверь, что item_docs соответствует внутренним item-id 0..n_items-1)"
        )
    vec = TfidfVectorizer(**tfidf_params)
    X = vec.fit_transform(item_docs).astype(np.float32)
    X = normalize(X, norm="l2", axis=1)

    # identity добавляет per-item feature (важно для LightFM, чтобы item мог иметь свои параметры)
    I = identity(pack["n_items"], format="csr", dtype=np.float32)
    return hstack([I, X], format="csr")

def make_recommender(pack, model, user_features, item_features, filter_seen=True, cache_topk=0):
    """
    Исправление ОЗУ:
    - НЕ кешируем полный scores (n_items,) для каждого пользователя.
    - опционально кешируем только top cache_topk (индексы), если cache_topk > 0.
    Для оффлайн-оценки ставь cache_topk=0.
    """
    all_items = pack["all_items"]
    user_train_items = pack["user_train_items"]
    cache = {}  # u -> np.int32 top indices (если cache_topk>0)

    def recommend_fn(u: int, K: int):
        K = int(K)
        if K <= 0:
            return np.array([], dtype=np.int32)

        if cache_topk and (u in cache):
            top = cache[u]
            return top[: min(K, top.shape[0])]

        scores = model.predict(
            user_ids=u,
            item_ids=all_items,
            user_features=user_features,
            item_features=item_features,
            num_threads=NUM_THREADS,
        )
        scores = np.asarray(scores, dtype=np.float32)

        if filter_seen:
            seen = user_train_items.get(u, None)
            if seen:
                scores = scores.copy()
                scores[list(seen)] = -np.inf

        kk = min(K, scores.shape[0])
        if cache_topk:
            kk = min(max(kk, int(cache_topk)), scores.shape[0])

        top = np.argpartition(-scores, kk - 1)[:kk]
        top = top[np.argsort(-scores[top])].astype(np.int32)

        if cache_topk:
            cache[u] = top  # сохраняем только индексы

        return top[: min(K, top.shape[0])]

    return recommend_fn

TFIDF_GRID = ParameterGrid({
    "ngram_range": [(1, 2)],
    "min_df": [2, 5],
    "max_df": [0.9],
    "max_features": [200_000, 300_000],
})

LFM_GRID = ParameterGrid({
    "loss": ["warp"],
    "no_components": [64],
    "learning_rate": [0.05, 0.02],
    "item_alpha": [1e-5],
    # user_alpha фиксируем = item_alpha
})

best = {"ndcg": -1.0, "tfidf": None, "lfm": None}
trial = 0
total = len(list(TFIDF_GRID)) * len(list(LFM_GRID))

for tf in TFIDF_GRID:
    tfidf_params = dict(
        lowercase=True,
        stop_words="english",
        norm="l2",
        sublinear_tf=True,
        **tf
    )

    # строим TF-IDF один раз на конфиг
    item_features_val = build_item_features(val_pack, item_docs_val, tfidf_params)
    user_features_val = identity(val_pack["n_users"], format="csr", dtype=np.float32)
    interactions_val = val_pack["R"].tocsr().astype(np.float32)

    for lf in LFM_GRID:
        trial += 1
        lfm_params = dict(
            random_state=SEED,
            loss=lf["loss"],
            no_components=lf["no_components"],
            learning_rate=lf["learning_rate"],
            item_alpha=lf["item_alpha"],
            user_alpha=lf["item_alpha"],
        )

        print(f"[{trial}/{total}] TFIDF={tf} | LFM={{k={lfm_params['no_components']}, alpha={lfm_params['item_alpha']}}}")

        model = LightFM(**lfm_params)
        model.fit(
            interactions_val,
            user_features=user_features_val,
            item_features=item_features_val,
            epochs=EPOCHS_TUNE,
            num_threads=NUM_THREADS
        )

        # ВАЖНО: cache_topk=0 для eval, иначе кеш будет пухнуть по числу уникальных пользователей
        rec_fn = make_recommender(val_pack, model, user_features_val, item_features_val,
                                  filter_seen=True, cache_topk=0)

        out = evaluate_next_item(val_u_small, val_i_small, rec_fn, ks=(TARGET_K,), name="LFM@VAL_tune")
        ndcg = out[TARGET_K][1]
        print("   ndcg@20:", ndcg)

        if ndcg > best["ndcg"]:
            best.update({"ndcg": ndcg, "tfidf": tfidf_params, "lfm": lfm_params})

        # чистим тяжелые объекты внутри trial
        del model, rec_fn, out
        gc.collect()

    # чистим после tf-конфига
    del item_features_val, user_features_val, interactions_val
    gc.collect()

print("\nBEST (tune):")
print("  tfidf:", {k: best["tfidf"][k] for k in ["min_df", "max_df", "max_features", "ngram_range"]})
print("  lfm  :", {k: best["lfm"][k] for k in ["no_components", "item_alpha", "user_alpha", "learning_rate", "loss"]})
print("  ndcg@20:", best["ndcg"])

# ---- полный VAL ----
item_features_val = build_item_features(val_pack, item_docs_val, best["tfidf"])
user_features_val = identity(val_pack["n_users"], format="csr", dtype=np.float32)
interactions_val = val_pack["R"].tocsr().astype(np.float32)

model_val = LightFM(**best["lfm"])
model_val.fit(
    interactions_val,
    user_features=user_features_val,
    item_features=item_features_val,
    epochs=EPOCHS_FINAL,
    num_threads=NUM_THREADS
)

rec_val = make_recommender(val_pack, model_val, user_features_val, item_features_val,
                           filter_seen=True, cache_topk=0)

val_out = evaluate_next_item(val_pack["eval_u"], val_pack["eval_i"], rec_val, ks=KS, name="LFM@VAL_full")
print_metrics("LFM@VAL_full", val_out, ks=KS)

# освободим VAL перед TEST
del item_features_val, user_features_val, interactions_val, model_val, rec_val, val_out
gc.collect()

# ---- TEST (с теми же гиперами) ----
item_features_test = build_item_features(test_pack, item_docs_test, best["tfidf"])
user_features_test = identity(test_pack["n_users"], format="csr", dtype=np.float32)
interactions_test = test_pack["R"].tocsr().astype(np.float32)

model_test = LightFM(**best["lfm"])
model_test.fit(
    interactions_test,
    user_features=user_features_test,
    item_features=item_features_test,
    epochs=EPOCHS_FINAL,
    num_threads=NUM_THREADS
)

rec_test = make_recommender(test_pack, model_test, user_features_test, item_features_test,
                            filter_seen=True, cache_topk=0)

test_out = evaluate_next_item(test_pack["eval_u"], test_pack["eval_i"], rec_test, ks=KS, name="LFM@TEST")
print_metrics("LFM@TEST", test_out, ks=KS)


[1/8] TFIDF={'max_df': 0.9, 'max_features': 200000, 'min_df': 2, 'ngram_range': (1, 2)} | LFM={k=64, alpha=1e-05}
[LFM@VAL_tune] short rec lists: 0/10000 (0.000)
   ndcg@20: 0.014358593860212869
[2/8] TFIDF={'max_df': 0.9, 'max_features': 200000, 'min_df': 2, 'ngram_range': (1, 2)} | LFM={k=64, alpha=1e-05}
[LFM@VAL_tune] short rec lists: 0/10000 (0.000)
   ndcg@20: 0.013892188099408185
[3/8] TFIDF={'max_df': 0.9, 'max_features': 200000, 'min_df': 5, 'ngram_range': (1, 2)} | LFM={k=64, alpha=1e-05}
[LFM@VAL_tune] short rec lists: 0/10000 (0.000)
   ndcg@20: 0.01252908776304771
[4/8] TFIDF={'max_df': 0.9, 'max_features': 200000, 'min_df': 5, 'ngram_range': (1, 2)} | LFM={k=64, alpha=1e-05}
[LFM@VAL_tune] short rec lists: 0/10000 (0.000)
   ndcg@20: 0.013992161530715006
[5/8] TFIDF={'max_df': 0.9, 'max_features': 300000, 'min_df': 2, 'ngram_range': (1, 2)} | LFM={k=64, alpha=1e-05}
[LFM@VAL_tune] short rec lists: 0/10000 (0.000)
   ndcg@20: 0.013352125313426569
[6/8] TFIDF={'max_df': 0.9

In [ ]:
# [1/96] TFIDF={'max_df': 0.9, 'max_features': 200000, 'min_df': 2, 'ngram_range': (1, 1)} | LFM={k=32, alpha=1e-06}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.014986062911817193
# [2/96] TFIDF={'max_df': 0.9, 'max_features': 200000, 'min_df': 2, 'ngram_range': (1, 1)} | LFM={k=64, alpha=1e-06}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.01414313521738967
# [3/96] TFIDF={'max_df': 0.9, 'max_features': 200000, 'min_df': 2, 'ngram_range': (1, 1)} | LFM={k=32, alpha=1e-06}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.014843427752592238
# [4/96] TFIDF={'max_df': 0.9, 'max_features': 200000, 'min_df': 2, 'ngram_range': (1, 1)} | LFM={k=64, alpha=1e-06}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.01525580128868959
# [5/96] TFIDF={'max_df': 0.9, 'max_features': 200000, 'min_df': 2, 'ngram_range': (1, 1)} | LFM={k=32, alpha=1e-05}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.014281228464676074
# [6/96] TFIDF={'max_df': 0.9, 'max_features': 200000, 'min_df': 2, 'ngram_range': (1, 1)} | LFM={k=64, alpha=1e-05}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.014463145491400686
# [7/96] TFIDF={'max_df': 0.9, 'max_features': 200000, 'min_df': 2, 'ngram_range': (1, 1)} | LFM={k=32, alpha=1e-05}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.014399583016419047
# [8/96] TFIDF={'max_df': 0.9, 'max_features': 200000, 'min_df': 2, 'ngram_range': (1, 1)} | LFM={k=64, alpha=1e-05}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.014898226606831865
# [9/96] TFIDF={'max_df': 0.9, 'max_features': 200000, 'min_df': 2, 'ngram_range': (1, 2)} | LFM={k=32, alpha=1e-06}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.014130894357542825
# [10/96] TFIDF={'max_df': 0.9, 'max_features': 200000, 'min_df': 2, 'ngram_range': (1, 2)} | LFM={k=64, alpha=1e-06}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.01421828095778219
# [11/96] TFIDF={'max_df': 0.9, 'max_features': 200000, 'min_df': 2, 'ngram_range': (1, 2)} | LFM={k=32, alpha=1e-06}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.014495041313010616
# [12/96] TFIDF={'max_df': 0.9, 'max_features': 200000, 'min_df': 2, 'ngram_range': (1, 2)} | LFM={k=64, alpha=1e-06}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.015325533459461192
# [13/96] TFIDF={'max_df': 0.9, 'max_features': 200000, 'min_df': 2, 'ngram_range': (1, 2)} | LFM={k=32, alpha=1e-05}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.014170402075287922
# [14/96] TFIDF={'max_df': 0.9, 'max_features': 200000, 'min_df': 2, 'ngram_range': (1, 2)} | LFM={k=64, alpha=1e-05}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.013965852734116456
# [15/96] TFIDF={'max_df': 0.9, 'max_features': 200000, 'min_df': 2, 'ngram_range': (1, 2)} | LFM={k=32, alpha=1e-05}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.014654212507114652
# [16/96] TFIDF={'max_df': 0.9, 'max_features': 200000, 'min_df': 2, 'ngram_range': (1, 2)} | LFM={k=64, alpha=1e-05}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.015255840185374538
# [17/96] TFIDF={'max_df': 0.9, 'max_features': 200000, 'min_df': 5, 'ngram_range': (1, 1)} | LFM={k=32, alpha=1e-06}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.014318203378039473
# [18/96] TFIDF={'max_df': 0.9, 'max_features': 200000, 'min_df': 5, 'ngram_range': (1, 1)} | LFM={k=64, alpha=1e-06}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.01450097460305298
# [19/96] TFIDF={'max_df': 0.9, 'max_features': 200000, 'min_df': 5, 'ngram_range': (1, 1)} | LFM={k=32, alpha=1e-06}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.014222345247294044
# [20/96] TFIDF={'max_df': 0.9, 'max_features': 200000, 'min_df': 5, 'ngram_range': (1, 1)} | LFM={k=64, alpha=1e-06}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.01514832380672224
# [21/96] TFIDF={'max_df': 0.9, 'max_features': 200000, 'min_df': 5, 'ngram_range': (1, 1)} | LFM={k=32, alpha=1e-05}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.014325679155225073
# [22/96] TFIDF={'max_df': 0.9, 'max_features': 200000, 'min_df': 5, 'ngram_range': (1, 1)} | LFM={k=64, alpha=1e-05}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.015128421457089746
# [23/96] TFIDF={'max_df': 0.9, 'max_features': 200000, 'min_df': 5, 'ngram_range': (1, 1)} | LFM={k=32, alpha=1e-05}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.014220108122111396
# [24/96] TFIDF={'max_df': 0.9, 'max_features': 200000, 'min_df': 5, 'ngram_range': (1, 1)} | LFM={k=64, alpha=1e-05}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.015011050410859131
# [25/96] TFIDF={'max_df': 0.9, 'max_features': 200000, 'min_df': 5, 'ngram_range': (1, 2)} | LFM={k=32, alpha=1e-06}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.013945668381026354
# [26/96] TFIDF={'max_df': 0.9, 'max_features': 200000, 'min_df': 5, 'ngram_range': (1, 2)} | LFM={k=64, alpha=1e-06}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.013543420361385054
# [27/96] TFIDF={'max_df': 0.9, 'max_features': 200000, 'min_df': 5, 'ngram_range': (1, 2)} | LFM={k=32, alpha=1e-06}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.014669476847248735
# [28/96] TFIDF={'max_df': 0.9, 'max_features': 200000, 'min_df': 5, 'ngram_range': (1, 2)} | LFM={k=64, alpha=1e-06}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.015453231754598937
# [29/96] TFIDF={'max_df': 0.9, 'max_features': 200000, 'min_df': 5, 'ngram_range': (1, 2)} | LFM={k=32, alpha=1e-05}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.01420803423636611
# [30/96] TFIDF={'max_df': 0.9, 'max_features': 200000, 'min_df': 5, 'ngram_range': (1, 2)} | LFM={k=64, alpha=1e-05}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.013870075374274787
# [31/96] TFIDF={'max_df': 0.9, 'max_features': 200000, 'min_df': 5, 'ngram_range': (1, 2)} | LFM={k=32, alpha=1e-05}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.014816988109984254
# [32/96] TFIDF={'max_df': 0.9, 'max_features': 200000, 'min_df': 5, 'ngram_range': (1, 2)} | LFM={k=64, alpha=1e-05}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.015016557997555562
# [33/96] TFIDF={'max_df': 0.9, 'max_features': 200000, 'min_df': 9, 'ngram_range': (1, 1)} | LFM={k=32, alpha=1e-06}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.014580280575641924
# [34/96] TFIDF={'max_df': 0.9, 'max_features': 200000, 'min_df': 9, 'ngram_range': (1, 1)} | LFM={k=64, alpha=1e-06}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.013968877888625945
# [35/96] TFIDF={'max_df': 0.9, 'max_features': 200000, 'min_df': 9, 'ngram_range': (1, 1)} | LFM={k=32, alpha=1e-06}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.014502840525744432
# [36/96] TFIDF={'max_df': 0.9, 'max_features': 200000, 'min_df': 9, 'ngram_range': (1, 1)} | LFM={k=64, alpha=1e-06}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.014699084191449984
# [37/96] TFIDF={'max_df': 0.9, 'max_features': 200000, 'min_df': 9, 'ngram_range': (1, 1)} | LFM={k=32, alpha=1e-05}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.014535076568315696
# [38/96] TFIDF={'max_df': 0.9, 'max_features': 200000, 'min_df': 9, 'ngram_range': (1, 1)} | LFM={k=64, alpha=1e-05}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.01459733169774545
# [39/96] TFIDF={'max_df': 0.9, 'max_features': 200000, 'min_df': 9, 'ngram_range': (1, 1)} | LFM={k=32, alpha=1e-05}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.014710721147521501
# [40/96] TFIDF={'max_df': 0.9, 'max_features': 200000, 'min_df': 9, 'ngram_range': (1, 1)} | LFM={k=64, alpha=1e-05}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.015097239631874433
# [41/96] TFIDF={'max_df': 0.9, 'max_features': 200000, 'min_df': 9, 'ngram_range': (1, 2)} | LFM={k=32, alpha=1e-06}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.013692607739739586
# [42/96] TFIDF={'max_df': 0.9, 'max_features': 200000, 'min_df': 9, 'ngram_range': (1, 2)} | LFM={k=64, alpha=1e-06}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.013384309057954347
# [43/96] TFIDF={'max_df': 0.9, 'max_features': 200000, 'min_df': 9, 'ngram_range': (1, 2)} | LFM={k=32, alpha=1e-06}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.014757829153485608
# [44/96] TFIDF={'max_df': 0.9, 'max_features': 200000, 'min_df': 9, 'ngram_range': (1, 2)} | LFM={k=64, alpha=1e-06}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.014801047474910607
# [45/96] TFIDF={'max_df': 0.9, 'max_features': 200000, 'min_df': 9, 'ngram_range': (1, 2)} | LFM={k=32, alpha=1e-05}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.014298217517708671
# [46/96] TFIDF={'max_df': 0.9, 'max_features': 200000, 'min_df': 9, 'ngram_range': (1, 2)} | LFM={k=64, alpha=1e-05}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.014375828475485125
# [47/96] TFIDF={'max_df': 0.9, 'max_features': 200000, 'min_df': 9, 'ngram_range': (1, 2)} | LFM={k=32, alpha=1e-05}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.01474370829671892
# [48/96] TFIDF={'max_df': 0.9, 'max_features': 200000, 'min_df': 9, 'ngram_range': (1, 2)} | LFM={k=64, alpha=1e-05}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.01485450839572593
# [49/96] TFIDF={'max_df': 0.9, 'max_features': 300000, 'min_df': 2, 'ngram_range': (1, 1)} | LFM={k=32, alpha=1e-06}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.013964072064291363
# [50/96] TFIDF={'max_df': 0.9, 'max_features': 300000, 'min_df': 2, 'ngram_range': (1, 1)} | LFM={k=64, alpha=1e-06}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.014402319350386585
# [51/96] TFIDF={'max_df': 0.9, 'max_features': 300000, 'min_df': 2, 'ngram_range': (1, 1)} | LFM={k=32, alpha=1e-06}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.014416538622908634
# [52/96] TFIDF={'max_df': 0.9, 'max_features': 300000, 'min_df': 2, 'ngram_range': (1, 1)} | LFM={k=64, alpha=1e-06}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.015201762170450148
# [53/96] TFIDF={'max_df': 0.9, 'max_features': 300000, 'min_df': 2, 'ngram_range': (1, 1)} | LFM={k=32, alpha=1e-05}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.014460583203943318
# [54/96] TFIDF={'max_df': 0.9, 'max_features': 300000, 'min_df': 2, 'ngram_range': (1, 1)} | LFM={k=64, alpha=1e-05}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.014351935121546847
# [55/96] TFIDF={'max_df': 0.9, 'max_features': 300000, 'min_df': 2, 'ngram_range': (1, 1)} | LFM={k=32, alpha=1e-05}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.014603094601528905
# [56/96] TFIDF={'max_df': 0.9, 'max_features': 300000, 'min_df': 2, 'ngram_range': (1, 1)} | LFM={k=64, alpha=1e-05}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.015174159845560121
# [57/96] TFIDF={'max_df': 0.9, 'max_features': 300000, 'min_df': 2, 'ngram_range': (1, 2)} | LFM={k=32, alpha=1e-06}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.01421619802503825
# [58/96] TFIDF={'max_df': 0.9, 'max_features': 300000, 'min_df': 2, 'ngram_range': (1, 2)} | LFM={k=64, alpha=1e-06}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.014654257645600379
# [59/96] TFIDF={'max_df': 0.9, 'max_features': 300000, 'min_df': 2, 'ngram_range': (1, 2)} | LFM={k=32, alpha=1e-06}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.014663909906350315
# [60/96] TFIDF={'max_df': 0.9, 'max_features': 300000, 'min_df': 2, 'ngram_range': (1, 2)} | LFM={k=64, alpha=1e-06}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.015034390390790856
# [61/96] TFIDF={'max_df': 0.9, 'max_features': 300000, 'min_df': 2, 'ngram_range': (1, 2)} | LFM={k=32, alpha=1e-05}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.01358097819802672
# [62/96] TFIDF={'max_df': 0.9, 'max_features': 300000, 'min_df': 2, 'ngram_range': (1, 2)} | LFM={k=64, alpha=1e-05}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.01472162627713241
# [63/96] TFIDF={'max_df': 0.9, 'max_features': 300000, 'min_df': 2, 'ngram_range': (1, 2)} | LFM={k=32, alpha=1e-05}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.015095921438254568
# [64/96] TFIDF={'max_df': 0.9, 'max_features': 300000, 'min_df': 2, 'ngram_range': (1, 2)} | LFM={k=64, alpha=1e-05}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.015073005556249663
# [65/96] TFIDF={'max_df': 0.9, 'max_features': 300000, 'min_df': 5, 'ngram_range': (1, 1)} | LFM={k=32, alpha=1e-06}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.013900770499819576
# [66/96] TFIDF={'max_df': 0.9, 'max_features': 300000, 'min_df': 5, 'ngram_range': (1, 1)} | LFM={k=64, alpha=1e-06}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.014796920596813016
# [67/96] TFIDF={'max_df': 0.9, 'max_features': 300000, 'min_df': 5, 'ngram_range': (1, 1)} | LFM={k=32, alpha=1e-06}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.014629061482810837
# [68/96] TFIDF={'max_df': 0.9, 'max_features': 300000, 'min_df': 5, 'ngram_range': (1, 1)} | LFM={k=64, alpha=1e-06}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.01507141056773766
# [69/96] TFIDF={'max_df': 0.9, 'max_features': 300000, 'min_df': 5, 'ngram_range': (1, 1)} | LFM={k=32, alpha=1e-05}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.014484747398407544
# [70/96] TFIDF={'max_df': 0.9, 'max_features': 300000, 'min_df': 5, 'ngram_range': (1, 1)} | LFM={k=64, alpha=1e-05}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.01462069889183613
# [71/96] TFIDF={'max_df': 0.9, 'max_features': 300000, 'min_df': 5, 'ngram_range': (1, 1)} | LFM={k=32, alpha=1e-05}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.014265689761710236
# [72/96] TFIDF={'max_df': 0.9, 'max_features': 300000, 'min_df': 5, 'ngram_range': (1, 1)} | LFM={k=64, alpha=1e-05}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.01498039427783882
# [73/96] TFIDF={'max_df': 0.9, 'max_features': 300000, 'min_df': 5, 'ngram_range': (1, 2)} | LFM={k=32, alpha=1e-06}
# [LFM@VAL_tune] short rec lists: 0/32656 (0.000)
#    ndcg@20: 0.013376707270344937
# [74/96] TFIDF={'max_df': 0.9, 'max_features': 300000, 'min_df': 5, 'ngram_range': (1, 2)} | LFM={k=64, alpha=1e-06}

In [ ]:
# Лучшие параметры (на текущий момент):
# TF-IDF:

# max_df: 0.9

# max_features: 200,000

# min_df: 5

# ngram_range: (1, 2)

# LFM (Latent Factor Model):

# k: 64

# alpha: 1e-06

# Основные выводы по результатам:
# k=64 против k=32: В большинстве случаев (как в №28, №12, №20) модель с k=64 показывает результаты лучше, чем с k=32. Это говорит о том, что для данных полезно более широкое скрытое пространство (Latent Space).

# ngram_range=(1, 2): Использование биграмм (сочетаний из двух слов) в связке с min_df=5 дало самый высокий скачок качества.

# Alpha: Значение 1e-06 (меньшая регуляризация) чаще встречается в топе результатов по сравнению с 1e-05, хотя разрыв не всегда критичен.

# Стабильность: Обратите внимание на итерацию №12 (ndcg@20: 0.015325) — она очень близка к лидеру, но использует min_df=2. Это значит, что модель довольно чувствительна к фильтрации редких слов в TF-IDF.

# Топ-3 комбинации:

# №28: 0.015453 (min_df: 5, k: 64, alpha: 1e-06)

# №12: 0.015325 (min_df: 2, k: 64, alpha: 1e-06)

# №16: 0.015255 (min_df: 2, k: 64, alpha: 1e-05)

## Matrix TF-IDF

In [ ]:
# !pip install optuna

In [ ]:
import numpy as np
from scipy.sparse import diags
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

def train_cb_tfidf(pack, item_docs, vec_params):
    """
    pack: val_pack или test_pack (из build_split_objects)
    item_docs: list[str] длины pack["n_items"] в порядке item index i=0..n_items-1
    vec_params: параметры TfidfVectorizer
    """
    vec = TfidfVectorizer(**vec_params)
    X_items = vec.fit_transform(item_docs).astype(np.float32)  # (n_items, n_terms)

    # user profiles = mean( TFIDF(items in history) ), затем L2-норм
    R = pack["R"].astype(np.float32)  # (n_users, n_items)
    prof = R @ X_items  # (n_users, n_terms)

    cnt = np.asarray(R.getnnz(axis=1)).astype(np.float32)
    cnt[cnt == 0] = 1.0
    prof = diags(1.0 / cnt) @ prof
    prof = normalize(prof, norm="l2", axis=1)

    model = {"vec": vec, "X_items": normalize(X_items, norm="l2", axis=1), "prof": prof}
    return model

def make_recommender(pack, model, filter_seen=True):
    X_items = model["X_items"]   # (n_items, n_terms), L2-норм
    prof = model["prof"]         # (n_users, n_terms), L2-норм

    user_train_items = pack["user_train_items"]
    cache = {}  # scores cache per user

    def recommend_fn(u: int, K: int):
        if u not in cache:
            scores = (prof[u] @ X_items.T).toarray().ravel()  # (n_items,)
            if filter_seen:
                seen = user_train_items.get(u, None)
                if seen:
                    scores[list(seen)] = -np.inf
            cache[u] = scores
        scores = cache[u]

        if K <= 0:
            return np.array([], dtype=np.int32)
        K = min(K, scores.shape[0])

        # topK без полной сортировки
        idx = np.argpartition(-scores, K-1)[:K]
        idx = idx[np.argsort(-scores[idx])]
        return idx.astype(np.int32)

    return recommend_fn


In [ ]:
# from sklearn.model_selection import ParameterGrid
# import numpy as np

KS = (5, 10, 20, 50)
# TARGET_K = 20

# # ускорение: тюним на подвыборке пользователей
# VAL_SAMPLE = 4000  # поставь 2000-5000
# rng = np.random.default_rng(SEED)
# idx = np.arange(len(val_pack["eval_u"]))
# if len(idx) > VAL_SAMPLE:
#     idx = rng.choice(idx, VAL_SAMPLE, replace=False)

# val_u_small = val_pack["eval_u"][idx]
# val_i_small = val_pack["eval_i"][idx]

# def score_on_val_fast(vec_params):
#     model = train_cb_tfidf(val_pack, item_docs_val, vec_params)
#     rec_fn = make_recommender(val_pack, model, filter_seen=True)
#     out = evaluate_next_item(val_u_small, val_i_small, rec_fn, ks=(TARGET_K,), name="CB@VAL_tune")
#     return out[TARGET_K][1]  # ndcg@20

# grid = ParameterGrid({
#     "min_df": [5, 10],
#     "max_df": [0.8, 1.0],
#     "max_features": [300_000],
# })

# best_params, best_ndcg = None, -1.0
# i = 0

# for p in grid:
#     i += 1
#     vec_params = {
#         "lowercase": True,
#         "stop_words": "english",
#         "norm": "l2",
#         "sublinear_tf": True,
#         "ngram_range": (1, 1),   # тюнинг без биграмм (быстро)
#         **p
#     }
#     print(f"[{i}/{len(grid)}] testing:", p)
#     ndcg = score_on_val_fast(vec_params)
#     print("   ndcg@20:", ndcg)

#     if ndcg > best_ndcg:
#         best_ndcg, best_params = ndcg, vec_params

# print("\nBest params:", best_params)
# print("Best tune NDCG@20:", best_ndcg)

# # честный прогон на полном VAL один раз
# model_val = train_cb_tfidf(val_pack, item_docs_val, best_params)
# rec_val = make_recommender(val_pack, model_val, filter_seen=True)
# val_out = evaluate_next_item(val_pack["eval_u"], val_pack["eval_i"], rec_val, ks=KS, name="CB@VAL_full")
# print_metrics("CB@VAL_full", val_out, ks=KS)

# 4] testing: {'max_df': 0.8, 'max_features': 300000, 'min_df': 5}
# [CB@VAL_tune] short rec lists: 0/4000 (0.000)
#    ndcg@20: 0.010600885258118645
# [2/4] testing: {'max_df': 0.8, 'max_features': 300000, 'min_df': 10}
# [CB@VAL_tune] short rec lists: 0/4000 (0.000)
#    ndcg@20: 0.010416051625984763
# [3/4] testing: {'max_df': 1.0, 'max_features': 300000, 'min_df': 5}
# [CB@VAL_tune] short rec lists: 0/4000 (0.000)
#    ndcg@20: 0.010660439531719057
# [4/4] testing: {'max_df': 1.0, 'max_features': 300000, 'min_df': 10}
# [CB@VAL_tune] short rec lists: 0/4000 (0.000)
#    ndcg@20: 0.010466082495226688

# Best params: {'lowercase': True, 'stop_words': 'english', 'norm': 'l2', 'sublinear_tf': True, 'ngram_range': (1, 1), 'max_df': 1.0, 'max_features': 300000, 'min_df': 5}
# Best tune NDCG@20: 0.010660439531719057

In [ ]:
# финальная модель: fit на item_docs_test (они собраны по train0_full и test_pack items)
final_model = train_cb_tfidf(test_pack, item_docs_test, vec_params={'lowercase': True, 'stop_words': 'english', 'norm': 'l2', 'sublinear_tf': True, 'ngram_range': (1, 1), 'max_df': 1.0, 'max_features': 300000, 'min_df': 5})
final_rec_fn = make_recommender(test_pack, final_model, filter_seen=True)

test_out = evaluate_next_item(test_pack["eval_u"], test_pack["eval_i"], final_rec_fn, ks=KS, name="CB@TEST")
print_metrics("CB@TEST", test_out, ks=KS)


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# для подбора гиперпараметров (VAL): fit на item_docs_val (построены по train_inner0_full)
tfidf_val = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1,2),
    min_df=2
)
X_items_val = tfidf_val.fit_transform(item_docs_val)

# для финального обучения/оценки (TEST): fit на item_docs_test (построены по train0_full)
tfidf_test = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1,2),
    min_df=2
)
X_items_test = tfidf_test.fit_transform(item_docs_test)

print("X_items_val:", X_items_val.shape, "| X_items_test:", X_items_test.shape)


X_items_val: (18699, 1856808) | X_items_test: (19983, 1991675)


In [ ]:
import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)

MIN_USER_INTERACTIONS = 5
K_TEST = 1
K_VAL  = 1

# 1) подготовка
df0 = df.copy()
df0["date"] = pd.to_datetime(df0["date"], errors="coerce")
df0 = df0.dropna(subset=["user_id","business_id","date"]).sort_values(["user_id","date"])

# user-filter
user_cnt = df0.groupby("user_id")["business_id"].size()
eligible_users = user_cnt[user_cnt >= MIN_USER_INTERACTIONS].index
df_cf = df0[df0["user_id"].isin(eligible_users)].copy()

# 2) split (как у тебя)
test_idx = df_cf.groupby("user_id").tail(K_TEST).index
test0  = df_cf.loc[test_idx, ["user_id","business_id","date"]].copy()
train0 = df_cf.drop(test_idx).loc[:, ["user_id","business_id","date"]].copy()

val_idx = train0.groupby("user_id").tail(K_VAL).index
val0 = train0.loc[val_idx, ["user_id","business_id","date"]].copy()
train_inner0 = train0.drop(val_idx).loc[:, ["user_id","business_id","date"]].copy()

print("Sizes:", "train_inner0", len(train_inner0), "val0", len(val0), "train0", len(train0), "test0", len(test0))

def seen_unseen_report(train_df, eval_df, name="EVAL"):
    """
    Seen/Unseen на уровне пользователя:
      seen_user = (u, item) уже встречалось в train для этого пользователя
      unseen_user = иначе
    Плюс:
      seen_global = item встречался в train у кого угодно
      unseen_global = иначе (item cold-start глобально)
    """
    # user-level history sets
    hist = train_df.groupby("user_id")["business_id"].apply(set).to_dict()

    # user-level seen/unseen
    seen_user = eval_df.apply(lambda r: r.business_id in hist.get(r.user_id, set()), axis=1)
    p_seen_user = seen_user.mean() if len(eval_df) else 0.0

    # global seen/unseen
    train_items = set(train_df["business_id"].values)
    seen_global = eval_df["business_id"].isin(train_items)
    p_seen_global = seen_global.mean() if len(eval_df) else 0.0

    print(f"\n[{name}] user-level:")
    print(f"  seen (target already in user's train history):   {p_seen_user:.4f} ({seen_user.sum()}/{len(eval_df)})")
    print(f"  unseen (target NEW for that user):               {(1-p_seen_user):.4f} ({(~seen_user).sum()}/{len(eval_df)})")

    print(f"[{name}] global-level:")
    print(f"  seen_global (item appears in train somewhere):   {p_seen_global:.4f} ({seen_global.sum()}/{len(eval_df)})")
    print(f"  unseen_global (item NOT in train at all):        {(1-p_seen_global):.4f} ({(~seen_global).sum()}/{len(eval_df)})")

# 3) отчёт по валидации и тесту
seen_unseen_report(train_inner0, val0,  name="VAL (vs train_inner0)")
seen_unseen_report(train0,       test0, name="TEST (vs train0)")

# 4) (опционально) "объединить": train+val как обучение и посмотреть тест
train_plus_val = pd.concat([train_inner0, val0], ignore_index=True)
seen_unseen_report(train_plus_val, test0, name="TEST (vs train_inner0+val0)")


Sizes: train_inner0 438469 val0 40711 train0 479180 test0 40711

[VAL (vs train_inner0)] user-level:
  seen (target already in user's train history):   0.0306 (1246/40711)
  unseen (target NEW for that user):               0.9694 (39465/40711)
[VAL (vs train_inner0)] global-level:
  seen_global (item appears in train somewhere):   0.9677 (39398/40711)
  unseen_global (item NOT in train at all):        0.0323 (1313/40711)

[TEST (vs train0)] user-level:
  seen (target already in user's train history):   0.0358 (1458/40711)
  unseen (target NEW for that user):               0.9642 (39253/40711)
[TEST (vs train0)] global-level:
  seen_global (item appears in train somewhere):   0.9680 (39407/40711)
  unseen_global (item NOT in train at all):        0.0320 (1304/40711)

[TEST (vs train_inner0+val0)] user-level:
  seen (target already in user's train history):   0.0358 (1458/40711)
  unseen (target NEW for that user):               0.9642 (39253/40711)
[TEST (vs train_inner0+val0)] global-l

In [ ]:
import numpy as np
import pandas as pd
from scipy.sparse import coo_matrix

SEED = 42
np.random.seed(SEED)

MIN_USER_INTERACTIONS = 5
MIN_ITEM_INTERACTIONS = 5   # для CF это критично; для CB можно ослабить, но пока оставим как в CF
K_TEST = 1
K_VAL  = 1

# ==============
# 1) PREP + SPLIT (как в CF)
# ==============
df0 = df.copy()
df0["date"] = pd.to_datetime(df0["date"], errors="coerce")
df0 = df0.dropna(subset=["user_id", "business_id", "date"]).sort_values(["user_id", "date"])

user_cnt = df0.groupby("user_id")["business_id"].size()
eligible_users = user_cnt[user_cnt >= MIN_USER_INTERACTIONS].index
df_cf = df0[df0["user_id"].isin(eligible_users)].copy()

test_idx = df_cf.groupby("user_id").tail(K_TEST).index
test0  = df_cf.loc[test_idx, ["user_id","business_id","date"]].copy()
train0 = df_cf.drop(test_idx).loc[:, ["user_id","business_id","date"]].copy()

val_idx = train0.groupby("user_id").tail(K_VAL).index
val0 = train0.loc[val_idx, ["user_id","business_id","date"]].copy()
train_inner0 = train0.drop(val_idx).loc[:, ["user_id","business_id","date"]].copy()

print("Split sizes:",
      "train_inner0", len(train_inner0),
      "val0", len(val0),
      "train0", len(train0),
      "test0", len(test0))


# ==============
# 2) BUILD PACK (как CF, но пригодно и для CB)
#    - item-filter по train (без утечки)
#    - индексация users/items по train
#    - матрица R (counts суммируются => weights)
# ==============
def build_pack(train_df, eval_df, min_item_interactions=5):
    # item-filter (как в CF)
    item_cnt = train_df.groupby("business_id")["user_id"].size()
    eligible_items = set(item_cnt[item_cnt >= min_item_interactions].index)

    eval_raw = len(eval_df)

    train_f = train_df[train_df["business_id"].isin(eligible_items)].copy()
    eval_f  = eval_df[eval_df["business_id"].isin(eligible_items)].copy()
    drop_item = eval_raw - len(eval_f)

    # индексация по train_f
    users = train_f["user_id"].unique()
    items = train_f["business_id"].unique()
    user2idx = {u:i for i,u in enumerate(users)}
    item2idx = {b:i for i,b in enumerate(items)}

    train_f["u"] = train_f["user_id"].map(user2idx).astype("int32")
    train_f["i"] = train_f["business_id"].map(item2idx).astype("int32")

    eval_f["u"] = eval_f["user_id"].map(user2idx)
    eval_f["i"] = eval_f["business_id"].map(item2idx)

    before_map = len(eval_f)
    eval_f = eval_f.dropna(subset=["u","i"]).copy()
    drop_user_missing = before_map - len(eval_f)

    eval_f["u"] = eval_f["u"].astype("int32")
    eval_f["i"] = eval_f["i"].astype("int32")

    n_users = len(users)
    n_items = len(items)

    # counts-as-weights: повторные (u,i) суммируются в CSR
    R = coo_matrix(
        (np.ones(len(train_f), dtype=np.float32),
         (train_f["u"].values, train_f["i"].values)),
        shape=(n_users, n_items)
    ).tocsr()

    # история пользователя (по train после фильтра)
    user_train_items = train_f.groupby("u")["i"].apply(lambda x: set(x.tolist())).to_dict()

    return {
        "R": R,
        "train_f": train_f,
        "eval_f": eval_f,
        "eval_u": eval_f["u"].values,
        "eval_i": eval_f["i"].values,
        "user_train_items": user_train_items,
        "user2idx": user2idx,
        "item2idx": item2idx,
        "n_users": n_users,
        "n_items": n_items,
        "eval_raw": eval_raw,
        "drop_item_filter": int(drop_item),
        "drop_user_missing": int(drop_user_missing),
    }

val_pack  = build_pack(train_inner0, val0,  min_item_interactions=MIN_ITEM_INTERACTIONS)
test_pack = build_pack(train0,       test0, min_item_interactions=MIN_ITEM_INTERACTIONS)


# ==============
# 3) ДИАГНОСТИКА: что выкинули и сколько "seen targets" в eval
# ==============
def pack_report(pack, name):
    print(f"\n[{name}] n_users={pack['n_users']} n_items={pack['n_items']}")
    print(f"[{name}] eval_raw={pack['eval_raw']} eval_kept={len(pack['eval_u'])}")
    print(f"[{name}] dropped_by_item_filter={pack['drop_item_filter']}")
    print(f"[{name}] dropped_by_user_missing={pack['drop_user_missing']}")

def seen_target_rate(pack, name):
    eval_u, eval_i = pack["eval_u"], pack["eval_i"]
    hist = pack["user_train_items"]
    seen = np.fromiter((int(i) in hist.get(int(u), set()) for u,i in zip(eval_u, eval_i)),
                       dtype=np.bool_, count=len(eval_u))
    rate = float(seen.mean()) if len(seen) else 0.0
    print(f"[{name}] target already in user's train history = {rate:.4f} ({int(seen.sum())}/{len(seen)})")
    return seen

pack_report(val_pack,  "VAL_PACK")
seen_target_rate(val_pack, "VAL_PACK")

pack_report(test_pack, "TEST_PACK")
seen_target_rate(test_pack, "TEST_PACK")


Split sizes: train_inner0 438469 val0 40711 train0 479180 test0 40711

[VAL_PACK] n_users=40593 n_items=18699
[VAL_PACK] eval_raw=40711 eval_kept=32656
[VAL_PACK] dropped_by_item_filter=8000
[VAL_PACK] dropped_by_user_missing=55
[VAL_PACK] target already in user's train history = 0.0315 (1030/32656)

[TEST_PACK] n_users=40666 n_items=19983
[TEST_PACK] eval_raw=40711 eval_kept=32032
[TEST_PACK] dropped_by_item_filter=8654
[TEST_PACK] dropped_by_user_missing=25
[TEST_PACK] target already in user's train history = 0.0378 (1212/32032)


array([False, False, False, ..., False, False, False])

In [ ]:
import numpy as np
import pandas as pd
import re, ast, json
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

# ---------- helpers: parsing ----------
def _clean_token(x: str) -> str:
    x = str(x).strip().lower()
    x = re.sub(r"[^\w=\-]+", "_", x)
    x = re.sub(r"_+", "_", x).strip("_")
    return x

def _safe_parse_obj(x):
    if x is None:
        return {}
    if isinstance(x, dict):
        return x
    if isinstance(x, float) and np.isnan(x):
        return {}
    s = str(x).strip()
    if s == "" or s.lower() in {"none", "nan"}:
        return {}
    try:
        return ast.literal_eval(s)
    except Exception:
        pass
    try:
        return json.loads(s)
    except Exception:
        return {}

def _flatten_attrs(obj, prefix="attr"):
    tokens = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            k2 = _clean_token(k)
            if not k2:
                continue
            if isinstance(v, dict):
                tokens.extend(_flatten_attrs(v, prefix=f"{prefix}_{k2}"))
            else:
                if isinstance(v, str):
                    vs = v.strip()
                    if vs.lower() in {"none", "false", ""}:
                        continue
                    if vs.lower() == "true":
                        tokens.append(f"{prefix}_{k2}")
                    else:
                        tokens.append(f"{prefix}_{k2}={_clean_token(vs)}")
                elif v is True:
                    tokens.append(f"{prefix}_{k2}")
                elif v in [False, None]:
                    continue
                else:
                    # numeric bucket
                    try:
                        vf = float(v)
                        if not np.isnan(vf):
                            tokens.append(f"{prefix}_{k2}=num")
                    except Exception:
                        pass
    return tokens

def business_to_text(row, use_numeric=True, use_attrs=True, use_cats=True):
    tokens = []

    if use_cats:
        cats = row.get("categories", None)
        if pd.notna(cats) and str(cats).strip() not in {"", "None", "nan"}:
            for c in str(cats).split(","):
                c2 = _clean_token(c)
                if c2:
                    tokens.append(f"cat_{c2}")

    if use_attrs:
        attrs = _safe_parse_obj(row.get("attributes", None))
        tokens.extend(_flatten_attrs(attrs, prefix="attr"))

    if use_numeric:
        sb = row.get("stars_business", None)
        if pd.notna(sb):
            try:
                sb = float(sb)
                sb = round(sb * 2) / 2.0
                tokens.append(f"stars_{str(sb).replace('.', '_')}")
            except Exception:
                pass

        rc = row.get("review_count", None)
        if pd.notna(rc):
            try:
                rc = float(rc)
                b = int(np.floor(np.log1p(max(rc, 0.0))))
                tokens.append(f"rc_log_{b}")
            except Exception:
                pass

        io = row.get("is_open", None)
        if pd.notna(io):
            try:
                io = int(io)
                tokens.append(f"is_open_{io}")
            except Exception:
                pass

    return " ".join(tokens)

# ---------- build biz_meta (one row per business) ----------
meta_cols = ["business_id", "attributes", "categories", "stars_business", "review_count", "is_open"]
meta_cols = [c for c in meta_cols if c in df0.columns]

biz_meta = (df0[meta_cols]
            .drop_duplicates("business_id", keep="first")
            .set_index("business_id"))

print("biz_meta rows:", biz_meta.shape[0], "cols:", list(biz_meta.columns))

def build_item_texts_for_pack(pack, biz_meta,
                              use_numeric=True, use_attrs=True, use_cats=True,
                              show_examples=3):
    train_f = pack["train_f"]
    n_items = pack["n_items"]

    # internal item index -> business_id
    i2biz = (train_f.drop_duplicates("i")[["i","business_id"]]
                  .set_index("i")["business_id"]
                  .to_dict())

    texts = np.empty(n_items, dtype=object)
    missing = 0
    empty = 0
    for i in range(n_items):
        bid = i2biz.get(i, None)
        if bid is None or bid not in biz_meta.index:
            texts[i] = ""
            missing += 1
            empty += 1
            continue
        row = biz_meta.loc[bid]
        t = business_to_text(row, use_numeric=use_numeric, use_attrs=use_attrs, use_cats=use_cats)
        texts[i] = t
        if t == "":
            empty += 1

    print(f"items={n_items} | missing_meta={missing} | empty_text={empty} ({empty/n_items:.3f})")

    # examples
    if show_examples and n_items > 0:
        idxs = np.random.choice(np.arange(n_items), size=min(show_examples, n_items), replace=False)
        for ii in idxs:
            print(f"example i={int(ii)} text[:200]: {texts[int(ii)][:200]}")

    return texts

# texts (на VAL_PACK достаточно — TEST_PACK будет аналогично)
val_texts = build_item_texts_for_pack(val_pack, biz_meta, use_numeric=True, use_attrs=True, use_cats=True)

# ---------- TF-IDF ----------
min_df = 2
max_features = 200_000

vec = TfidfVectorizer(
    ngram_range=(1, 1),          # без биграмм
    min_df=min_df,
    max_features=max_features,
    token_pattern=r"(?u)\b[0-9a-zA-Z_=\-]+\b"
)
X_item_val = vec.fit_transform(val_texts)
X_item_val = normalize(X_item_val, norm="l2")

print("\nTFIDF VAL:")
print("  X_item_val shape:", X_item_val.shape)
print("  vocab size:", len(vec.vocabulary_))
print("  nnz:", X_item_val.nnz)


biz_meta rows: 48208 cols: ['attributes', 'categories', 'stars_business', 'review_count', 'is_open']
items=18699 | missing_meta=0 | empty_text=0 (0.000)
example i=3070 text[:200]: cat_bars cat_sports_bars cat_chicken_wings cat_american_traditional cat_nightlife cat_restaurants attr_alcohol=u_full_bar attr_ambience=u_divey_false_u_hipster_false_u_casual_true_u_touristy_false_u_t
example i=17529 text[:200]: cat_health_markets cat_grocery cat_organic_stores cat_food cat_specialty_food attr_businessparking=garage_false_street_false_validated_false_lot_true_valet_false attr_restaurantsdelivery stars_3_5 rc_
example i=15598 text[:200]: cat_soup cat_salad cat_restaurants cat_buffets attr_alcohol=u_none attr_ambience=u_divey_false_u_hipster_false_u_casual_true_u_touristy_false_u_trendy_false_u_intimate_false_u_romantic_false_u_classy_

TFIDF VAL:
  X_item_val shape: (18699, 1907)
  vocab size: 1907
  nnz: 361455


In [ ]:
import numpy as np
from sklearn.preprocessing import normalize

KS = (5, 10, 20, 50)

def _ndcg_from_rank(rank: int) -> float:
    return 1.0 / np.log2(rank + 2)

def _ap_from_rank(rank: int) -> float:
    return 1.0 / (rank + 1)

def evaluate_next_item(eval_u, eval_i, recommend_fn, ks=KS, name="model"):
    ks = tuple(sorted(ks))
    maxK = ks[-1]
    rec = {k: 0.0 for k in ks}
    ndcg = {k: 0.0 for k in ks}
    mp = {k: 0.0 for k in ks}
    n = len(eval_u)
    short_lists = 0

    for u, pos in zip(eval_u, eval_i):
        recs = recommend_fn(int(u), int(maxK))
        recs = np.asarray(recs, dtype=np.int32) if recs is not None else np.array([], dtype=np.int32)
        if len(recs) < maxK:
            short_lists += 1

        rank_map = {int(it): r for r, it in enumerate(recs)}
        rpos = rank_map.get(int(pos), None)

        for k in ks:
            if rpos is None or rpos >= k:
                continue
            rec[k] += 1.0
            ndcg[k] += _ndcg_from_rank(rpos)
            mp[k] += _ap_from_rank(rpos)

    out = {k: (rec[k]/n, ndcg[k]/n, mp[k]/n) for k in ks}
    print(f"[{name}] short rec lists: {short_lists}/{n} ({short_lists/max(1,n):.3f})")
    return out

def print_metrics(name, out, ks=KS):
    print(f"\n=== {name} ===")
    for k in ks:
        r, n, m = out[k]
        print(f"K={k:2d} | Recall@{k}: {r:.4f} | NDCG@{k}: {n:.4f} | MAP@{k}: {m:.4f}")

def filter_strict_unseen(pack, name="PACK"):
    eval_u, eval_i = pack["eval_u"], pack["eval_i"]
    hist = pack["user_train_items"]
    mask = np.fromiter((int(i) not in hist.get(int(u), set()) for u,i in zip(eval_u, eval_i)),
                       dtype=np.bool_, count=len(eval_u))
    print(f"[{name}] STRICT_UNSEEN kept {int(mask.sum())}/{len(mask)} (dropped {int((~mask).sum())})")
    newp = dict(pack)
    newp["eval_u"] = eval_u[mask]
    newp["eval_i"] = eval_i[mask]
    return newp

def fit_cb_tfidf_for_pack(pack, biz_meta, *, min_df=2, max_features=200_000,
                          use_numeric=True, use_attrs=True, use_cats=True):
    # build item texts
    train_f = pack["train_f"]
    n_items = pack["n_items"]
    i2biz = (train_f.drop_duplicates("i")[["i","business_id"]]
                  .set_index("i")["business_id"]
                  .to_dict())

    texts = np.empty(n_items, dtype=object)
    for i in range(n_items):
        bid = i2biz[i]
        row = biz_meta.loc[bid]
        texts[i] = business_to_text(row, use_numeric=use_numeric, use_attrs=use_attrs, use_cats=use_cats)

    vec = TfidfVectorizer(
        ngram_range=(1, 1),
        min_df=int(min_df),
        max_features=int(max_features),
        token_pattern=r"(?u)\b[0-9a-zA-Z_=\-]+\b"
    )
    X_item = vec.fit_transform(texts)
    X_item = normalize(X_item, norm="l2")

    R = pack["R"].tocsr().astype(np.float32)
    U = R @ X_item  # counts act as weights
    U = normalize(U, norm="l2")

    X_item_T = X_item.T.tocsr()
    user_train_items = pack["user_train_items"]

    def recommend_cb(u, K):
        seen = user_train_items.get(int(u), set())
        scores = U[int(u)].dot(X_item_T).toarray().ravel()
        if len(seen) > 0:
            scores[list(seen)] = -np.inf
        kk = min(int(K), scores.size - len(seen))
        if kk <= 0:
            return np.array([], dtype=np.int32)
        topk = np.argpartition(-scores, kk-1)[:kk]
        topk = topk[np.argsort(-scores[topk])]
        return topk.astype(np.int32)

    return recommend_cb, vec, X_item, U

# ---- TRAIN+EVAL VAL ----
val_pack_strict = filter_strict_unseen(val_pack, "VAL_PACK")

rec_val, vec_val, X_item_val, U_val = fit_cb_tfidf_for_pack(
    val_pack, biz_meta,
    min_df=2, max_features=200_000,
    use_numeric=True, use_attrs=True, use_cats=True
)

out_val_raw = evaluate_next_item(val_pack["eval_u"], val_pack["eval_i"], rec_val, ks=KS, name="VAL/CB-TFIDF RAW")
print_metrics("VAL/CB-TFIDF RAW", out_val_raw, ks=KS)

out_val_str = evaluate_next_item(val_pack_strict["eval_u"], val_pack_strict["eval_i"], rec_val, ks=KS, name="VAL/CB-TFIDF STRICT_UNSEEN")
print_metrics("VAL/CB-TFIDF STRICT_UNSEEN", out_val_str, ks=KS)

# ---- TRAIN+EVAL TEST ----
test_pack_strict = filter_strict_unseen(test_pack, "TEST_PACK")

rec_test, vec_test, X_item_test, U_test = fit_cb_tfidf_for_pack(
    test_pack, biz_meta,
    min_df=2, max_features=200_000,
    use_numeric=True, use_attrs=True, use_cats=True
)

out_test_raw = evaluate_next_item(test_pack["eval_u"], test_pack["eval_i"], rec_test, ks=KS, name="TEST/CB-TFIDF RAW")
print_metrics("TEST/CB-TFIDF RAW", out_test_raw, ks=KS)

out_test_str = evaluate_next_item(test_pack_strict["eval_u"], test_pack_strict["eval_i"], rec_test, ks=KS, name="TEST/CB-TFIDF STRICT_UNSEEN")
print_metrics("TEST/CB-TFIDF STRICT_UNSEEN", out_test_str, ks=KS)


[VAL_PACK] STRICT_UNSEEN kept 31626/32656 (dropped 1030)
[VAL/CB-TFIDF RAW] short rec lists: 0/32656 (0.000)

=== VAL/CB-TFIDF RAW ===
K= 5 | Recall@5: 0.0023 | NDCG@5: 0.0014 | MAP@5: 0.0011
K=10 | Recall@10: 0.0044 | NDCG@10: 0.0021 | MAP@10: 0.0014
K=20 | Recall@20: 0.0074 | NDCG@20: 0.0028 | MAP@20: 0.0016
K=50 | Recall@50: 0.0143 | NDCG@50: 0.0042 | MAP@50: 0.0018
[VAL/CB-TFIDF STRICT_UNSEEN] short rec lists: 0/31626 (0.000)

=== VAL/CB-TFIDF STRICT_UNSEEN ===
K= 5 | Recall@5: 0.0023 | NDCG@5: 0.0014 | MAP@5: 0.0011
K=10 | Recall@10: 0.0046 | NDCG@10: 0.0021 | MAP@10: 0.0014
K=20 | Recall@20: 0.0077 | NDCG@20: 0.0029 | MAP@20: 0.0016
K=50 | Recall@50: 0.0148 | NDCG@50: 0.0043 | MAP@50: 0.0018
[TEST_PACK] STRICT_UNSEEN kept 30820/32032 (dropped 1212)
[TEST/CB-TFIDF RAW] short rec lists: 0/32032 (0.000)

=== TEST/CB-TFIDF RAW ===
K= 5 | Recall@5: 0.0029 | NDCG@5: 0.0018 | MAP@5: 0.0014
K=10 | Recall@10: 0.0044 | NDCG@10: 0.0023 | MAP@10: 0.0016
K=20 | Recall@20: 0.0074 | NDCG@20: 0.